# Mage-Flow-Edit-Turbo on Kaggle TPU v5e-8 — Production Demo

This notebook runs the repository's native JAX/TPU edit pipeline. It does not replace model inference with post-processing. The production-qualified baseline is **512×512 / q256 / 4 steps**; 768×768 / q256 is validated scaling; 1024×1024 / q128 remains experimental.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path

print(sys.version)
print("KAGGLE_KERNEL_RUN_TYPE=", os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

## 1. Configuration

Attach or restore the converted JAX/Orbax artifact before running. Do not put tokens in this notebook. Set `MAGE_FLOW_EDIT_SOURCE` to a reference image path. `ARTIFACT_ROOT` must contain `text_encoder/`, `checkpoints/{text_encoder,transformer,vae}`, `manifests/`, and `rope/`.

In [ ]:
REPO_ROOT = Path(os.environ.get("MAGE_FLOW_EDIT_REPO", "/kaggle/working/Mage-Flow-Edit-Turbo-On-TPU-v5e8"))
ARTIFACT_ROOT = Path(os.environ.get("MAGE_FLOW_EDIT_ARTIFACT", "/kaggle/input/mage-flow-edit-jax-artifact"))
source_env = os.environ.get("MAGE_FLOW_EDIT_SOURCE")
assert source_env, "Set MAGE_FLOW_EDIT_SOURCE to the reference image path before running."
SOURCE_IMAGE = Path(source_env)
OUTPUT_DIR = Path(os.environ.get("MAGE_FLOW_EDIT_OUTPUT", "/kaggle/working/mage-flow-edit-demo"))
PROMPT = os.environ.get("MAGE_FLOW_EDIT_PROMPT", "Transform the scene while preserving the subject and composition.")
RESOLUTION = int(os.environ.get("MAGE_FLOW_EDIT_RESOLUTION", "512"))
SEEDS = os.environ.get("MAGE_FLOW_EDIT_SEEDS", "42,43,44,45")
print(REPO_ROOT, ARTIFACT_ROOT, SOURCE_IMAGE, OUTPUT_DIR, RESOLUTION, SEEDS, sep="\n")

## 2. Preflight

The demo fails early if the repository, source image, or required artifact layout is missing.

In [ ]:
required = [
    REPO_ROOT / "bootstrap/08_run_tpu_edit.py",
    SOURCE_IMAGE,
    ARTIFACT_ROOT / "text_encoder",
    ARTIFACT_ROOT / "checkpoints/text_encoder",
    ARTIFACT_ROOT / "checkpoints/transformer",
    ARTIFACT_ROOT / "checkpoints/vae",
    ARTIFACT_ROOT / "manifests",
    ARTIFACT_ROOT / "rope",
]
missing=[str(p) for p in required if not p.exists()]
assert not missing, "Missing required paths:\n" + "\n".join(missing)
print("PREFLIGHT=PASS")

In [ ]:
env=os.environ.copy()
env.update({
    "PJRT_DEVICE":"TPU", "TPU_ACCELERATOR_TYPE":"v5litepod-8",
    "TPU_CHIPS_PER_HOST_BOUNDS":"2,4,1", "TPU_HOST_BOUNDS":"1,1,1",
    "TPU_PROCESS_ADDRESSES":"local", "TPU_SKIP_MDS_QUERY":"1",
    "TPU_WORKER_HOSTNAMES":"localhost", "TPU_WORKER_ID":"0",
    "KERAS_BACKEND":"jax", "USE_TORCH_XLA":"0",
})
code='import jax; print("TPU_DEVICES", len(jax.devices())); print(jax.devices())'
subprocess.run([sys.executable,'-c',code],check=True,env=env)

## 3. Run the native Edit-Turbo pipeline

In [ ]:
cmd=[sys.executable, str(REPO_ROOT/'bootstrap/08_run_tpu_edit.py'),
     '--artifact-root',str(ARTIFACT_ROOT),'--image',str(SOURCE_IMAGE),'--prompt',PROMPT,
     '--output',str(OUTPUT_DIR),'--resolution',str(RESOLUTION),'--topology','4x2','--seeds',SEEDS]
print(' '.join(map(str,cmd)))
subprocess.run(cmd,check=True,env=env)

## 4. Verify outputs

A successful run must report `status=PASS` and produce all requested PNGs.

In [ ]:
summary=json.loads((OUTPUT_DIR/'generation_summary.json').read_text())
assert summary['status']=='PASS', summary
for p in summary['pngs']:
    assert Path(p).is_file(), p
print(json.dumps(summary,indent=2))

In [ ]:
from IPython.display import display
from PIL import Image
for p in summary['pngs']:
    print(Path(p).name)
    display(Image.open(p))

## Release evidence

The approved visual showcase includes a separately reviewed **768×768 q256 seed 52** snow-leopard result. This notebook defaults to the production-qualified 512 path so a fresh public run does not silently claim 768 as production-qualified.

In [ ]:
manifest_path = REPO_ROOT / "release-evidence/PUBLISH_MANIFEST.json"
manifest=json.loads(manifest_path.read_text())
snow=manifest["selected"][0]
print("approved_showcase_seed=", snow["seed"])
print("approved_showcase_sha256=", snow["after_sha256"])
assert snow["seed"] == 52
assert snow["after_sha256"] == "84f8da54ee56deb2649ded7f55ccc6c6150bb7cfea4b70fe0cb3ec8a7b365aaf"
